# Traceprop-LLM -- Stage 2: speed sweep (Pythia-1B) + LDS sweep (from-scratch) + backdoor detection

Supersedes the earlier single-backend design. Three separate experiments now run on the settings where each actually produces signal, per compute-cost analysis (see below):

1. **Speed + measured storage** (`exp31`) stays on **Pythia-1B**, tracked scope `{last-1, last-6, all}` -- this is where overhead differences actually show up, and it's the number that matters for the paper's systems claim.
2. **LDS / compression-quality** (`exp35`) moves to the **from-scratch tiny LoRA transformer** (`--backend tiny`), tracked scope `{last-1, all}`. Pythia-1B/SST-2 is a LoRA fine-tune of an already-pretrained model on a task it partly solves -- the paper's own §3.4 data shows LDS near zero there for every method, so a LogIX-vs-Traceprop LDS comparison in that regime would likely just be noise, and it's expensive (100 Pythia-1B subset retrains is hours of L4 time). The tiny/synthetic setting has real signal (LDS ~0.65-0.70, confirmed in `results/exp27_tiny_tiny-clf.json`) and is CPU-fast (~2 hours total for both scope points at n_subsets=200, vs. an estimated ~13 hours on Pythia-1B for a comparably-powered sweep).
3. **Backdoor detection** (`exp37`, PRIMARY) stays on **Pythia-1B**, ~1% overhead setting -- plants a trigger phrase + flipped label into K training examples, then for triggered test inputs attributes the (backdoored) prediction back to training data. A loss-ranking baseline cannot do this (backdoored examples have LOW loss once learned), so this is a genuine test of attribution, not just outlier detection. Mislabel detection via self-influence is kept as a SECONDARY result, reported next to loss-ranking and gradient-norm baselines.

**Framing note for the paper**: `exp35`'s LDS comparison scores BOTH tools on post-hoc, final-checkpoint gradients (isolates the compression scheme, not an inline-quality claim). Inline-quality evidence comes from item 9 (§3.4) and the backdoor experiment, both Traceprop-only -- already stated in Related Work and the §3.3 pending-marker comment in `main.tex`.

**Compute estimate** (using the real measured Pythia-1B step time, 424ms/step at batch=16/seq=64, from `results/exp25_hf_pythia-1b_track1.json`):
- exp31 at full settings (steps=200, repeats=20, both init configs) all three scopes: **~5.7 hours** -- too long. Reduced to full settings ONLY at track=1 (matches Table 1's existing methodology) and steps=100/repeats=10 at track={6,0} (secondary sweep points): **~2.8 hours**.
- exp35 on the tiny backend, n_subsets=200, both scope points: **~1 hour** (CPU; the original Pythia-1B/SST-2 plan was ~13.25 hours for a comparable subset count).
- exp37 at real settings (n_train=2000, 3 epochs): well under an hour.
- **Total: roughly 4 hours**, down from ~19 hours for the notebook as originally written. Still substantial -- if a session disconnects partway, results already written to Drive (`/content/drive/MyDrive/traceprop_runs/`) survive; rerun only the missing pieces.

## Setup: pin versions, mount Drive, assert L4, clone repo

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert 'L4' in torch.cuda.get_device_name(0), (
    f"expected an L4, got {torch.cuda.get_device_name(0)} -- Table 1/2, the sweep, and this "
    f"session's runs are all L4-only; a different GPU would make the numbers incomparable. "
    f"Reconnect and request an L4 runtime."
)

!pip -q install "transformers==4.44.2" "peft==0.13.2" "accelerate==0.34.2" "numpy<2" datasets scipy scikit-learn
!pip -q install --ignore-requires-python logix-ai

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/traceprop_runs', exist_ok=True)

In [ ]:
import getpass
TOKEN = getpass.getpass('GitHub token: ').strip()
url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git'
!git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
%cd /content/Traceprop
!pip -q install -e .
%cd /content/Traceprop/experiments

## Step 0: 1-repeat GPU dry run (do this before anything else)

Only exp31 and exp37 touch the GPU (Pythia-1B); exp35 moved to the CPU-fast tiny backend and was already validated locally, no GPU dry run needed for it. Gradient validation stays ON. If either of these fails, fix it here -- minute 5, not hour 2.

In [ ]:
!python exp31_logix_comparison.py --backend hf --model EleutherAI/pythia-1b --device cuda \
    --steps 5 --repeats 1 --warmup 1 --pca_cov_steps 2 --track 1 \
    --out /tmp/dryrun_exp31.json --force

In [ ]:
!python exp37_planted_detection.py --backend hf --data sst2 --model EleutherAI/pythia-1b --device cuda \
    --n_train 32 --n_trigger_test 16 --plant_frac 0.1 --mislabel_frac 0.1 --epochs 1 --batch 8 --track 1 \
    --out /tmp/dryrun_exp37.json --force

**If both printed `gradient validation OK` (exp31) and finished without error, proceed. If anything failed, stop and fix it before running the real sweep below.**

## Step 1: exp31 speed sweep on Pythia-1B -- full settings at track=1, reduced at track={6,0}

Full settings (steps=200, repeats=20) at track=1 match Table 1's existing methodology exactly. track={6,0} use steps=100/repeats=10 (secondary sweep points, less statistical power but ~4x cheaper) -- raise back to full settings if there's time budget left after everything else finishes.

In [ ]:
sweep_settings = [
    (1, 'last1', 200, 20),
    (6, 'last6', 100, 10),
    (0, 'all',   100, 10),
]
for track, label, steps, repeats in sweep_settings:
    print(f'=== exp31 track={track} ({label}), steps={steps} repeats={repeats} ===')
    !python exp31_logix_comparison.py --backend hf --model EleutherAI/pythia-1b --device cuda \
        --steps {steps} --repeats {repeats} --warmup 10 --pca_cov_steps 20 --track {track} \
        --out results/exp31_pythia1b_track{track}.json --force
    !cp results/exp31_pythia1b_track{track}.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

## Step 2: exp35 LDS sweep on the from-scratch tiny LoRA transformer -- {last-1, all}

CPU is fine here (fast regardless); n_subsets=200 (lower-powered than the paper's canonical 500 for §3.4, raise if there's time). n_blocks=2 (default) means track=0 ("all") already covers both blocks, so track={1,0} is the full 2-point sweep this model supports.

In [ ]:
for track, label in [(1, 'last1'), (0, 'all')]:
    print(f'=== exp35 (tiny backend) track={track} ({label}) ===')
    !python exp35_logix_lds.py --backend tiny --device cpu \
        --n_train 400 --n_test 100 --n_subsets 200 --subset_frac 0.5 --epochs 3 \
        --track {track} --lora_init pca \
        --out results/exp35_tiny_track{track}.json --force
    !cp results/exp35_tiny_track{track}.json results/exp35_tiny_track{track}_raw.npz \
        /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

## Step 3: backdoor detection (exp37, PRIMARY) on Pythia-1B, ~1% overhead setting

Mislabel detection (self-influence vs. loss-ranking vs. gradient-norm baselines) runs alongside as the secondary result, same training pass.

In [ ]:
!python exp37_planted_detection.py --backend hf --data sst2 --model EleutherAI/pythia-1b --device cuda \
    --n_train 2000 --n_trigger_test 200 --plant_frac 0.05 --mislabel_frac 0.05 \
    --epochs 3 --batch 16 --proj_dim 512 --track 1 \
    --out results/exp37_pythia1b_track1.json --force
!cp results/exp37_pythia1b_track1.json results/exp37_pythia1b_track1_raw.npz \
    /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

## Read everything back

In [ ]:
import json

print('=== exp31: overhead + measured bytes, by tracked scope (Pythia-1B) ===')
for track, label, *_ in sweep_settings:
    d = json.load(open(f'results/exp31_pythia1b_track{track}.json'))
    sm = d['storage_matching']
    print(f"  track={label}: tracked_modules={d['tracked_modules_count']} "
          f"measured_bytes/ex={sm['logix_bytes_per_example_measured']} "
          f"matching_proj_dim={sm['traceprop_proj_dim_to_match']}")
    for cfg_name, cfg in d['configs'].items():
        print(f"    {cfg_name}: overhead={cfg['overhead_pct_median']}% +/- {cfg['overhead_pct_std']}%"
              + (f" covariance_pass={cfg['covariance_pass_s']}s" if cfg.get('covariance_pass_s') else ''))
    gv = d['gradient_validation']
    print(f"    gradient_validation: worst_cosine={min(g['worst_cosine'] for g in gv):.6f} "
          f"worst_nonuniformity={max(g['scale_nonuniformity'] for g in gv):.6f}")

print('\n=== exp35: LDS, by tracked scope (from-scratch tiny transformer) ===')
for track, label in [(1, 'last1'), (0, 'all')]:
    d = json.load(open(f'results/exp35_tiny_track{track}.json'))
    sm = d['storage_matching']
    print(f"  track={label}: measured_bytes/ex={sm['logix_bytes_per_example_measured']} "
          f"matched_proj_dim={sm['traceprop_proj_dim_matched']}")
    for k, v in d['lds'].items():
        print(f"    {k:<22} {v['mean']:+.4f} +/- {v['std']:.4f}")
    gv = d['gradient_validation']
    print(f"    gradient_validation: worst_cosine={gv['worst_cosine']:.6f} "
          f"nonuniformity={gv['scale_nonuniformity']:.6f}")

print('\n=== exp37: backdoor detection (primary) + mislabel detection (secondary), Pythia-1B ===')
d = json.load(open('results/exp37_pythia1b_track1.json'))
print(f"  inline overhead={d['overhead_pct']}%")
pb = d['primary_backdoor']
print(f"  PRIMARY backdoor: success_rate={pb['backdoor_success_rate']:.4f} "
      f"attribution_AUC={pb['attribution_auc']:.4f} "
      f"precision@{d['k_backdoor']}={pb['attribution_precision_at_k']:.4f} "
      f"(random_AUC={pb['random_baseline_auc']:.4f})")
print(f"  SECONDARY mislabel (vs. loss/grad_norm baselines):")
for k, v in d['secondary_mislabel'].items():
    print(f"    {k:<15} AUC={v['auc']:.4f} precision@{d['k_mislabel']}={v['precision_at_k']:.4f}")

## Bring results back to Claude Code

Paste the printed summary above. It will: (1) fill in the §3.3 storage-matched draft from `docs/mlsys/LOGIX_OUTCOME_DRAFTS.md` using the real numbers, (2) add the scope-sweep table to `main.tex` (fills in the currently-pending "overhead vs. tracked parameters" claim, using exp31's Pythia-1B data), (3) add the LDS-vs-scope table using exp35's tiny-backend data, explicitly framed as a final-checkpoint compression-quality comparison, (4) write up the backdoor detection result as the paper's headline attribution-works evidence, with mislabel detection plus the loss/grad-norm baselines as a secondary table.